# Baseline: GRU4Rec

Entrena el `GRU4RecModel` de la implementación oficial en PyTorch con el protocolo de LatentR3 (Adam, grilla de lr x weight decay, BCE con negativos aleatorios, contexto de hasta 10 ítems) y mide HR y NDCG @5 y @10 por id y por título.

El entrenamiento está en `gru4rec_baseline.py` y el protocolo común (datos y métricas) en `../baseline_comun.py`, para correr las combinaciones de la grilla en 6 procesos en paralelo. Este notebook carga y verifica los datos, lanza la grilla, arma las tablas y compara con la Tabla 1 del paper. Además de BCE (paper) corre las variantes de referencia CE catálogo, BCE todas las posiciones y CE todas las posiciones.

Fuente, configuración, resultados y problemas conocidos (incluido el estado de CE todas las posiciones): [README.md](README.md). No escribe archivos.


## Importaciones

In [1]:
import os
import time
import itertools
import multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed
import numpy as np
import pandas as pd
import torch

E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48


In [2]:
N_PROCESOS = 6   # combinaciones que se entrenan a la vez; todas comparten la GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

cuda AMD Radeon RX 9070 XT


## Datos y código oficial

In [3]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


In [4]:
import sys
base = 'data_amazon_2018' if IN_COLAB else os.path.join('..', '..', 'repo_latentr3', 'data')
codigo = os.path.abspath('GRU4Rec_PyTorch_Official' if IN_COLAB else os.path.join('..', '..', 'GRU4Rec_PyTorch_Official'))
# baseline_comun.py está en notebooks/ en local y en el directorio de trabajo en Colab. Los procesos de
# la grilla heredan sys.path, así que también lo encuentran
comun = os.path.abspath('.' if IN_COLAB else '..')
if comun not in sys.path:
    sys.path.insert(0, comun)
names = {"CDs_and_Vinyl": "cd_data",
         "Musical_Instruments": "music_data",
         "Toys_and_Games": "toys_data",
         "Video_Games": "games_data"}

if not os.path.isfile(os.path.join(codigo, "gru4rec_pytorch.py")):
    raise FileNotFoundError(f"No se encontró gru4rec_pytorch.py en {codigo}. "
                            "Descargar https://github.com/hidasib/GRU4Rec_PyTorch_Official en esa carpeta "
                            "y hacer checkout del commit d1fc311.")
for modulo in ['gru4rec_baseline.py', os.path.join(comun, 'baseline_comun.py')]:
    if not os.path.isfile(modulo):
        raise FileNotFoundError(f'No se encontró {os.path.abspath(modulo)}.')

import gru4rec_baseline as gb
gb.inicializar_proceso(codigo)   # verifica el sha256 de gru4rec_pytorch.py antes de importarlo

In [15]:
resumen = {}
for categoria, nombre in names.items():
    datos, titulos, grupos = gb.cargar_categoria(base, categoria)
    # Cada fila de train aparece una vez por época
    tamanos = [len(y) for _, y in gb.lotes(grupos["train"], gb.BATCH_SIZE, np.random.default_rng(0))]
    assert sum(tamanos) == len(datos["train"])
    resumen[nombre] = {**{split: len(df) for split, df in datos.items()}, "ítems": len(titulos)}
print(resumen)

{'cd_data': {'train': 49251, 'valid': 6156, 'test': 6157, 'ítems': 5841}, 'music_data': {'train': 66500, 'valid': 8312, 'test': 8313, 'ítems': 5030}, 'toys_data': {'train': 53898, 'valid': 6737, 'test': 6738, 'ítems': 6299}, 'games_data': {'train': 75175, 'valid': 9397, 'test': 9397, 'ítems': 5308}}


## Métricas

`hit`, `ndcg`, `dedup_titulos` y `evaluar` están en `../baseline_comun.py`: HR y NDCG @5 y @10 **por id** y **por título** (como `calc.py`), con un solo ítem relevante. `evaluar` verifica que las métricas por título nunca sean menores que por id. El criterio por título se explica en el README de `notebooks/`.

## Modelo GRU4Rec

**No lo especifica el paper:** capa de 100, batch de 64, sin dropout y embedding compartido (valores por defecto oficiales); un negativo uniforme por fila; hasta 200 épocas con early stopping por NDCG@10 (id) en `valid`, paciencia 10; ranking sobre todo el catálogo de `info/`, igual que los métodos con LLM.

**Del código oficial se usa solo `GRU4RecModel`** (sus métodos `embed`, `hidden_step` y `score_items`), sin modificarlo. En `gru4rec_baseline.py`:
- `cargar_categoria` y `lotes`: una muestra por fila de `train` (historial de ≤10 ítems → `item_id`), en lotes de filas con el mismo largo de historial. Son las mismas muestras y el mismo contexto que usan los métodos con LLM.
- `codificar`: recorre el historial desde un estado oculto en cero y devuelve el último estado. La retropropagación pasa por todos los pasos (≤10), en vez de truncarse a uno como en el oficial.
- `perdida_bce` (la del paper: el objetivo contra un negativo uniforme) y `perdida_ce` (referencia: softmax sobre todo el catálogo).
- `codificar_todas` y `perdida_bce_todas` (referencia): el estado oculto después de cada paso, y BCE en cada posición contra el ítem siguiente, con un negativo uniforme por posición.
- `perdida_ce_todas` (referencia): softmax sobre todo el catálogo en cada posición.
- `entrenar`: Adam con weight decay y embeddings densos (Adam no acepta gradientes dispersos), early stopping en `valid` y vuelta a la mejor época.
- `correr_combinacion`: entrena una combinación con las semillas fijadas y devuelve su NDCG@10 en `valid`, las épocas corridas y sus métricas en `test`.

## Grilla y evaluación

Para cada variante y dataset se entrenan las 9 combinaciones de lr ∈ {1e-2, 1e-3, 1e-4} × weight decay ∈ {1e-4, 1e-5, 1e-6}. Esta celda corre BCE (paper) y CE catálogo, 72 en total, repartidas en `N_PROCESOS` procesos que comparten la GPU. Cada combinación fija sus semillas antes de empezar, así que el resultado no depende del orden ni del paralelismo (salvo el ruido propio de la GPU).

Las métricas de `test` se calculan en cada proceso, pero la configuración se elige **solo** con el mejor NDCG@10 (id) en `valid`; ante un empate gana la primera en el orden de la grilla. La variante comparable con la Tabla 1 es **BCE (paper)** con el criterio **por título**.

In [16]:
LRS = [1e-2, 1e-3, 1e-4]
WEIGHT_DECAYS = [1e-4, 1e-5, 1e-6]

VARIANTES = ['BCE (paper)', 'CE catálogo']   # "BCE todas las posiciones" corre en su propia celda, al final
combinaciones = list(itertools.product(VARIANTES, names, LRS, WEIGHT_DECAYS))
corridas = [None] * len(combinaciones)
inicio = time.time()
contexto = multiprocessing.get_context("spawn")   # CUDA no admite procesos creados con fork
with ProcessPoolExecutor(N_PROCESOS, mp_context=contexto, initializer=gb.inicializar_proceso,
                         initargs=(codigo,)) as pool:
    futuros = {pool.submit(gb.correr_combinacion, base, categoria, variante, lr, wd): i
               for i, (variante, categoria, lr, wd) in enumerate(combinaciones)}
    for n, futuro in enumerate(as_completed(futuros), 1):
        r = futuro.result()
        corridas[futuros[futuro]] = r
        print(f"[{n}/{len(combinaciones)}] {r['variante']} {names[r['categoria']]} lr={r['lr']} wd={r['weight_decay']}: "
              f"NDCG@10 valid {r['NDCG@10 valid']:.4f} (época {r['época']}) [{(time.time() - inicio) / 60:.1f} min]",
              flush=True)

corridas = pd.DataFrame(corridas)   # las 72 combinaciones, en el orden de la grilla
corridas.insert(1, "dataset", corridas.pop("categoria").map(names))
# Mejor configuración por variante y dataset, elegida solo con valid
resultados = corridas.loc[corridas.groupby(["variante", "dataset"], sort=False)["NDCG@10 valid"].idxmax()]
resultados = resultados.reset_index(drop=True)
resultados.round(4)

E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48


[1/72] BCE (paper) cd_data lr=0.01 wd=1e-06: NDCG@10 valid 0.0124 (época 1) [0.5 min]
[2/72] BCE (paper) cd_data lr=0.0001 wd=0.0001: NDCG@10 valid 0.0074 (época 17) [1.5 min]
[3/72] BCE (paper) cd_data lr=0.01 wd=0.0001: NDCG@10 valid 0.0188 (época 29) [1.6 min]
[4/72] BCE (paper) cd_data lr=0.001 wd=0.0001: NDCG@10 valid 0.0203 (época 37) [1.9 min]
[5/72] BCE (paper) cd_data lr=0.01 wd=1e-05: NDCG@10 valid 0.0167 (época 49) [2.4 min]
[6/72] BCE (paper) cd_data lr=0.0001 wd=1e-05: NDCG@10 valid 0.0078 (época 15) [2.4 min]
[7/72] BCE (paper) cd_data lr=0.0001 wd=1e-06: NDCG@10 valid 0.0078 (época 15) [2.5 min]
[8/72] BCE (paper) music_data lr=0.01 wd=1e-05: NDCG@10 valid 0.0541 (época 9) [3.3 min]
[9/72] BCE (paper) music_data lr=0.01 wd=0.0001: NDCG@10 valid 0.0675 (época 34) [3.9 min]
[10/72] BCE (paper) cd_data lr=0.001 wd=1e-06: NDCG@10 valid 0.0493 (época 98) [4.1 min]
[11/72] BCE (paper) music_data lr=0.01 wd=1e-06: NDCG@10 valid 0.0573 (época 33) [4.5 min]
[12/72] BCE (paper) mu

,variante,dataset,lr,weight_decay,época,NDCG@10 valid,HR@5 id,NDCG@5 id,HR@10 id,NDCG@10 id,HR@5 título,NDCG@5 título,HR@10 título,NDCG@10 título
0,BCE (paper),cd_data,0.001,0.0000,195,0.0638,0.0754,0.0555,0.1020,0.0641,0.0797,0.0580,0.1095,0.0676
1,BCE (paper),music_data,0.001,0.0000,60,0.0892,0.0811,0.0685,0.1007,0.0748,0.0812,0.0686,0.1009,0.0749
2,BCE (paper),toys_data,0.001,0.0000,131,0.0526,0.0499,0.0331,0.0703,0.0396,0.0499,0.0331,0.0703,0.0396
3,BCE (paper),games_data,0.001,0.0000,79,0.0512,0.0384,0.0257,0.0629,0.0336,0.0384,0.0257,0.0630,0.0336
4,CE catálogo,cd_data,0.010,0.0001,7,0.0755,0.0884,0.0716,0.1117,0.0791,0.0940,0.0753,0.1200,0.0836
5,CE catálogo,music_data,0.001,0.0000,5,0.1093,0.0948,0.0825,0.1134,0.0885,0.0949,0.0826,0.1137,0.0886
6,CE catálogo,toys_data,0.001,0.0000,11,0.0740,0.0698,0.0539,0.0850,0.0588,0.0699,0.0541,0.0853,0.0590
7,CE catálogo,games_data,0.001,0.0001,17,0.0760,0.0597,0.0413,0.0897,0.0510,0.0597,0.0413,0.0897,0.0510


## Comparación con la Tabla 1 del paper

GRU4Rec según el paper, junto a la variante BCE por id y por título, y la variante CE por título (referencia).

In [ ]:
PAPER = {"toys_data": [0.0417, 0.0564, 0.0305, 0.0352], "cd_data": [0.0481, 0.0669, 0.0365, 0.0425],
         "games_data": [0.0322, 0.0517, 0.0207, 0.0270], "music_data": [0.0766, 0.0960, 0.0630, 0.0692]}
METRICAS = ["HR@5", "HR@10", "NDCG@5", "NDCG@10"]

def fila(variante, nombre, criterio):
    r = resultados[(resultados["variante"] == variante) & (resultados["dataset"] == nombre)].iloc[0]
    return [r[f"{m} {criterio}"] for m in METRICAS]

tabla = []
for nombre in names.values():
    for fuente, valores in [("Paper", PAPER[nombre]),
                            ("BCE (paper), id", fila("BCE (paper)", nombre, "id")),
                            ("BCE (paper), título", fila("BCE (paper)", nombre, "título")),
                            ("CE catálogo, título", fila("CE catálogo", nombre, "título"))]:
        tabla.append({"dataset": nombre, "fuente": fuente, **dict(zip(METRICAS, valores))})
pd.DataFrame(tabla).round(4)

,dataset,fuente,HR@5,HR@10,NDCG@5,NDCG@10
0,cd_data,Paper,0.0481,0.0669,0.0365,0.0425
1,cd_data,"BCE (paper), id",0.0754,0.1020,0.0555,0.0641
2,cd_data,"BCE (paper), título",0.0797,0.1095,0.0580,0.0676
3,cd_data,"CE catálogo, título",0.0940,0.1200,0.0753,0.0836
4,music_data,Paper,0.0766,0.0960,0.0630,0.0692
5,music_data,"BCE (paper), id",0.0811,0.1007,0.0685,0.0748
6,music_data,"BCE (paper), título",0.0812,0.1009,0.0686,0.0749
7,music_data,"CE catálogo, título",0.0949,0.1137,0.0826,0.0886
8,toys_data,Paper,0.0417,0.0564,0.0305,0.0352
9,toys_data,"BCE (paper), id",0.0499,0.0703,0.0331,0.0396


: 

## Variante de referencia: BCE en todas las posiciones

Usa las mismas filas de `train`, pero cada posición del historial predice el ítem siguiente con su contexto (en la última, el objetivo de la fila), contra un negativo uniforme por posición. Es la supervisión del GRU4Rec oficial, donde cada evento de la sesión predice el siguiente. La diferencia con BCE (paper) mide cuánto le cuesta a GRU4Rec entrenar solo en la última posición (decisión común [D1](../README.md#decisiones-comunes-d1d17): una muestra por fila de `train`, con la pérdida en el último paso). No se compara con la Tabla 1.

No depende de la celda de la grilla: se puede correr después de las celdas de datos. Muestra su propia mejor configuración por dataset, elegida solo con `valid`.


In [5]:
variante = 'BCE todas las posiciones'
LRS = [1e-2, 1e-3, 1e-4]
WEIGHT_DECAYS = [1e-4, 1e-5, 1e-6]
combinaciones_todas = list(itertools.product(names, LRS, WEIGHT_DECAYS))
filas = [None] * len(combinaciones_todas)
inicio = time.time()
contexto = multiprocessing.get_context('spawn')   # CUDA no admite procesos creados con fork
with ProcessPoolExecutor(N_PROCESOS, mp_context=contexto, initializer=gb.inicializar_proceso,
                         initargs=(codigo,)) as pool:
    futuros = {pool.submit(gb.correr_combinacion, base, categoria, variante, lr, wd): i
               for i, (categoria, lr, wd) in enumerate(combinaciones_todas)}
    for n, futuro in enumerate(as_completed(futuros), 1):
        r = futuro.result()
        filas[futuros[futuro]] = r
        print(f"[{n}/{len(combinaciones_todas)}] {variante} {names[r['categoria']]} lr={r['lr']} wd={r['weight_decay']}: "
              f"NDCG@10 valid {r['NDCG@10 valid']:.4f} (época {r['época']} de {r['épocas corridas']}) "
              f"[{(time.time() - inicio) / 60:.1f} min]", flush=True)

corridas_todas = pd.DataFrame(filas)   # las 36 combinaciones, en el orden de la grilla
corridas_todas.insert(1, 'dataset', corridas_todas.pop('categoria').map(names))
# Mejor configuración por dataset, elegida solo con valid. lr y weight decay en notación científica,
# porque con round(4) 1e-5 y 1e-6 se verían como 0
mejor = corridas_todas.loc[corridas_todas.groupby('dataset', sort=False)['NDCG@10 valid'].idxmax()]
mejor = mejor.reset_index(drop=True)
mejor.assign(lr=mejor['lr'].map('{:.0e}'.format),
             weight_decay=mejor['weight_decay'].map('{:.0e}'.format)).round(4)

E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48


[1/36] BCE todas las posiciones cd_data lr=0.01 wd=0.0001: NDCG@10 valid 0.0241 (época 19 de 29) [2.0 min]
[2/36] BCE todas las posiciones cd_data lr=0.01 wd=1e-05: NDCG@10 valid 0.0379 (época 20 de 30) [2.1 min]
[3/36] BCE todas las posiciones cd_data lr=0.01 wd=1e-06: NDCG@10 valid 0.0281 (época 23 de 33) [2.4 min]
[4/36] BCE todas las posiciones cd_data lr=0.001 wd=0.0001: NDCG@10 valid 0.0228 (época 37 de 47) [3.2 min]
[5/36] BCE todas las posiciones cd_data lr=0.0001 wd=1e-06: NDCG@10 valid 0.0070 (época 8 de 18) [3.6 min]
[6/36] BCE todas las posiciones music_data lr=0.01 wd=0.0001: NDCG@10 valid 0.0606 (época 12 de 22) [4.9 min]
[7/36] BCE todas las posiciones cd_data lr=0.0001 wd=0.0001: NDCG@10 valid 0.0078 (época 34 de 44) [4.9 min]
[8/36] BCE todas las posiciones music_data lr=0.01 wd=1e-05: NDCG@10 valid 0.0737 (época 10 de 20) [5.2 min]
[9/36] BCE todas las posiciones cd_data lr=0.0001 wd=1e-05: NDCG@10 valid 0.0083 (época 39 de 49) [5.3 min]
[10/36] BCE todas las posicion

,variante,dataset,lr,weight_decay,época,épocas corridas,NDCG@10 valid,HR@5 id,NDCG@5 id,HR@10 id,NDCG@10 id,HR@5 título,NDCG@5 título,HR@10 título,NDCG@10 título
0,BCE todas las posiciones,cd_data,1e-03,1e-06,135,145,0.0639,0.0827,0.0632,0.0994,0.0686,0.0861,0.0655,0.1046,0.0715
1,BCE todas las posiciones,music_data,1e-03,1e-05,66,76,0.0931,0.0842,0.0711,0.1022,0.0769,0.0842,0.0711,0.1022,0.0769
2,BCE todas las posiciones,toys_data,1e-03,1e-05,126,136,0.0636,0.0545,0.0382,0.0793,0.0462,0.0545,0.0382,0.0794,0.0462
3,BCE todas las posiciones,games_data,1e-03,1e-05,82,92,0.0545,0.0420,0.0278,0.0666,0.0357,0.0420,0.0278,0.0666,0.0357


## Variante de referencia: CE en todas las posiciones

Combina CE catálogo con la supervisión en todas las posiciones: softmax sobre todo el catálogo en cada posición del historial, con media sobre las posiciones. Es la variante más cercana al GRU4Rec oficial que permite el protocolo: el oficial supervisa cada evento con softmax contra el resto del lote y negativos muestreados. No reproduce su contexto de sesión completa ni Adagrad (ver [README.md](README.md)). No se compara con la Tabla 1.

Como la anterior, no depende de la celda de la grilla y muestra su propia mejor configuración por dataset, elegida solo con `valid`. La corrida guardada fue interrumpida (ver Problemas conocidos en el README).


In [5]:
variante = 'CE todas las posiciones'
LRS = [1e-2, 1e-3, 1e-4]
WEIGHT_DECAYS = [1e-4, 1e-5, 1e-6]
combinaciones_ce_todas = list(itertools.product(names, LRS, WEIGHT_DECAYS))
filas = [None] * len(combinaciones_ce_todas)
inicio = time.time()
contexto = multiprocessing.get_context('spawn')   # CUDA no admite procesos creados con fork
with ProcessPoolExecutor(N_PROCESOS, mp_context=contexto, initializer=gb.inicializar_proceso,
                         initargs=(codigo,)) as pool:
    futuros = {pool.submit(gb.correr_combinacion, base, categoria, variante, lr, wd): i
               for i, (categoria, lr, wd) in enumerate(combinaciones_ce_todas)}
    for n, futuro in enumerate(as_completed(futuros), 1):
        r = futuro.result()
        filas[futuros[futuro]] = r
        print(f"[{n}/{len(combinaciones_ce_todas)}] {variante} {names[r['categoria']]} lr={r['lr']} wd={r['weight_decay']}: "
              f"NDCG@10 valid {r['NDCG@10 valid']:.4f} (época {r['época']} de {r['épocas corridas']}) "
              f"[{(time.time() - inicio) / 60:.1f} min]", flush=True)

corridas_ce_todas = pd.DataFrame(filas)   # las 36 combinaciones, en el orden de la grilla
corridas_ce_todas.insert(1, 'dataset', corridas_ce_todas.pop('categoria').map(names))
# Mejor configuración por dataset, elegida solo con valid. lr y weight decay en notación científica,
# porque con round(4) 1e-5 y 1e-6 se verían como 0
mejor = corridas_ce_todas.loc[corridas_ce_todas.groupby('dataset', sort=False)['NDCG@10 valid'].idxmax()]
mejor = mejor.reset_index(drop=True)
mejor.assign(lr=mejor['lr'].map('{:.0e}'.format),
             weight_decay=mejor['weight_decay'].map('{:.0e}'.format)).round(4)

E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48


KeyboardInterrupt: 